In [1]:
import pandas as pd
import numpy as np
from sklearn.isotonic import IsotonicRegression

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)



In [2]:
train_df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/train.csv")
test_df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/test.csv")
sub_df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/sample_submission.csv")

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("sub_df shape:", sub_df.shape)




train shape: (49232, 14)
test shape: (5474, 9)
sub_df shape: (2384, 2)


In [3]:
def get_num_unique(train_df, test_df, col):
    all_df = pd.concat([train_df, test_df], ignore_index=True)
    num_unique_train = len(train_df[col].unique())
    num_unique_test = len(test_df[col].unique())
    num_unique_all = len(all_df[col].unique())
    return num_unique_train, num_unique_test, num_unique_all


def add_count(df, col):
    if isinstance(col, str):
        aggs = (
            df.groupby(col, as_index=False)
            .size()
            .rename(columns={"size": f"{col}_count"})
        )
        df = df.merge(aggs, on=col, how="inner")
        return df
    else:
        count_name = "_".join(col) + "_count"
        aggs = (
            df.groupby(col, as_index=False).size().rename(columns={"size": count_name})
        )
        df = df.merge(aggs, on=col, how="inner")
        return df




In [4]:
hypotheses = []



In [5]:
num_unique_train, num_unique_test, num_unique_all = get_num_unique(
    train_df, test_df, "site_id"
)
print(f"num_unique_train: {num_unique_train}")
print(f"num_unique_test: {num_unique_test}")
print(f"num_unique_all: {num_unique_all}")
hypothesis = num_unique_all == 2
print(f"hyposthesis: {hypothesis}")
hypotheses.append(hypothesis)



num_unique_train: 2
num_unique_test: 2
num_unique_all: 2
hyposthesis: True


In [6]:
num_unique_train, num_unique_test, num_unique_all = get_num_unique(
    train_df, test_df, "patient_id"
)
print(f"num_unique_train: {num_unique_train}")
print(f"num_unique_test: {num_unique_test}")
print(f"num_unique_all: {num_unique_all}")
hypothesis = num_unique_train + num_unique_test == num_unique_all
print(f"hyposthesis: {hypothesis}")
hypotheses.append(hypothesis)



num_unique_train: 10721
num_unique_test: 1192
num_unique_all: 11913
hyposthesis: True


In [7]:
num_unique_train, num_unique_test, num_unique_all = get_num_unique(
    train_df, test_df, "image_id"
)
print(f"num_unique_train: {num_unique_train}")
print(f"num_unique_test: {num_unique_test}")
print(f"num_unique_all: {num_unique_all}")
hypothesis = num_unique_train + num_unique_test == num_unique_all
print(f"hyposthesis: {hypothesis}")
hypotheses.append(hypothesis)



num_unique_train: 49232
num_unique_test: 5474
num_unique_all: 54706
hyposthesis: True


In [8]:
num_unique_train, num_unique_test, num_unique_all = get_num_unique(
    train_df, test_df, "laterality"
)
print(f"num_unique_train: {num_unique_train}")
print(f"num_unique_test: {num_unique_test}")
print(f"num_unique_all: {num_unique_all}")
hypothesis = num_unique_test == 2
print(f"hyposthesis: {hypothesis}")
hypotheses.append(hypothesis)



num_unique_train: 2
num_unique_test: 2
num_unique_all: 2
hyposthesis: True


In [9]:
num_unique_train, num_unique_test, num_unique_all = get_num_unique(
    train_df, test_df, "machine_id"
)
print(f"num_unique_train: {num_unique_train}")
print(f"num_unique_test: {num_unique_test}")
print(f"num_unique_all: {num_unique_all}")
hypothesis = num_unique_train != num_unique_all
print(f"hyposthesis: {hypothesis}")
hypotheses.append(hypothesis)



num_unique_train: 10
num_unique_test: 9
num_unique_all: 10
hyposthesis: False


In [10]:
num_unique_train, num_unique_test, num_unique_all = get_num_unique(
    train_df, test_df, "view"
)
print(f"num_unique_train: {num_unique_train}")
print(f"num_unique_test: {num_unique_test}")
print(f"num_unique_all: {num_unique_all}")
hypothesis = num_unique_all == 6
print(f"hyposthesis: {hypothesis}")
hypotheses.append(hypothesis)



num_unique_train: 6
num_unique_test: 4
num_unique_all: 6
hyposthesis: True


In [11]:
temp_df = add_count(test_df, "patient_id").drop_duplicates("patient_id")
print(temp_df.head())
hypothesis = temp_df.patient_id_count.min() >= 4
print(f"hyposthesis: {hypothesis}")
hypotheses.append(hypothesis)



    site_id  patient_id   image_id laterality view   age  implant  machine_id  \
0         2       10116  472095321          L   CC  67.0        0          29   
4         1       10130  388811999          L  MLO  71.0        0          49   
11        1       10188  109667135          L   CC  40.0        0          49   
17        1       10200  134267365          L   CC  70.0        0          49   
25        2       10219  657488461          L  MLO  65.0        0          21   

   prediction_id  patient_id_count  
0        10116_L                 4  
4        10130_L                 7  
11       10188_L                 6  
17       10200_L                 8  
25       10219_L                 4  
hyposthesis: True


In [12]:
len1 = len(test_df.drop_duplicates(["patient_id"]))
len2 = len(test_df.drop_duplicates(["patient_id", "site_id"]))
print(f"len1: {len1}")
print(f"len2: {len2}")
hypothesis = len1 == len2
print(f"hyposthesis: {hypothesis}")
hypotheses.append(hypothesis)



len1: 1192
len2: 1192
hyposthesis: True


In [13]:
len1 = len(train_df.drop_duplicates(["patient_id"]))
len2 = len(train_df.drop_duplicates(["patient_id", "machine_id"]))
print(f"len1: {len1}")
print(f"len2: {len2}")
hypothesis = len1 == len2
print(f"hyposthesis: {hypothesis}")
print(train_df[train_df.patient_id == 22637].head())



len1: 10721
len2: 10721
hyposthesis: True
Empty DataFrame
Columns: [site_id, patient_id, image_id, laterality, view, age, cancer, biopsy, invasive, BIRADS, implant, density, machine_id, difficult_negative_case]
Index: []


In [14]:
print(hypotheses)
print(all(hypotheses))




[True, True, True, True, False, True, True, True]
False


In [15]:
def probabilistic_f1(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_prob = np.clip(y_prob, 0.0, 1.0)

    pTP = np.sum(y_prob * y_true)
    pFP = np.sum(y_prob * (1.0 - y_true))

    denom_prec = pTP + pFP
    pPrecision = pTP / denom_prec if denom_prec > 0 else 0.0

    P = np.sum(y_true)  # TP + FN
    pRecall = pTP / P if P > 0 else 0.0

    denom_f1 = pPrecision + pRecall
    return (2.0 * pPrecision * pRecall / denom_f1) if denom_f1 > 0 else 0.0


def make_oof_predictions_for_alpha(df, group_cols, alpha, n_folds=5, seed=RANDOM_SEED):
    df = df.copy()

    df["prediction_id"] = (
        df["patient_id"].astype(str) + "-" + df["laterality"].astype(str)
    )

    patients = df["patient_id"].drop_duplicates().to_numpy()
    rng_local = np.random.default_rng(seed)
    rng_local.shuffle(patients)
    folds = np.array_split(patients, n_folds)

    global_mean = float(df["cancer"].mean())

    oof_img = pd.Series(index=df.index, dtype=np.float64)

    for fold_patients in folds:
        is_val = df["patient_id"].isin(fold_patients)
        tr = df.loc[~is_val]
        va = df.loc[is_val]

        rates = tr.groupby(group_cols, as_index=False).agg(
            mean_cancer=("cancer", "mean"), cnt=("cancer", "size")
        )
        rates["smoothed"] = (
            rates["mean_cancer"] * rates["cnt"] + global_mean * alpha
        ) / (rates["cnt"] + alpha)

        va2 = va.copy()
        va2["_row_id_"] = va2.index.astype(np.int64)
        va2 = va2.merge(rates[group_cols + ["smoothed"]], on=group_cols, how="left")

        pred = va2["smoothed"].fillna(global_mean).astype(np.float64).clip(0.0, 1.0)
        oof_img.loc[va2["_row_id_"].values] = pred.values

    tmp = df[["prediction_id", "cancer"]].copy()
    tmp["oof_pred"] = oof_img.values

    y_true_pid = tmp.groupby("prediction_id", as_index=False)["cancer"].max()
    y_pred_pid = tmp.groupby("prediction_id", as_index=False)["oof_pred"].mean()

    merged = y_true_pid.merge(y_pred_pid, on="prediction_id", how="inner").sort_values(
        "prediction_id"
    )
    return (
        merged["cancer"].astype(np.float64).values,
        merged["oof_pred"].astype(np.float64).values,
    )


group_cols = ["site_id", "laterality", "view"]
alpha_grid = [1.0, 5.0, 10.0, 20.0, 40.0, 80.0]

best_alpha = None
best_score = -1.0

for a in alpha_grid:
    y_true_pid, oof_pred_pid = make_oof_predictions_for_alpha(
        train_df, group_cols, alpha=a, n_folds=5, seed=RANDOM_SEED
    )
    score = probabilistic_f1(y_true_pid, oof_pred_pid)
    print(f"alpha={a:>6}: OOF pF1 proxy (prediction_id-level)={score:.6f}")
    if score > best_score:
        best_score = score
        best_alpha = a

print("Selected alpha:", best_alpha, "with OOF pF1 proxy:", best_score)



alpha=   1.0: OOF pF1 proxy (prediction_id-level)=0.020114
alpha=   5.0: OOF pF1 proxy (prediction_id-level)=0.020112
alpha=  10.0: OOF pF1 proxy (prediction_id-level)=0.020112
alpha=  20.0: OOF pF1 proxy (prediction_id-level)=0.020112
alpha=  40.0: OOF pF1 proxy (prediction_id-level)=0.020113
alpha=  80.0: OOF pF1 proxy (prediction_id-level)=0.020116
Selected alpha: 80.0 with OOF pF1 proxy: 0.02011611232409737


In [16]:
y_true_pid, oof_pred_pid = make_oof_predictions_for_alpha(
    train_df, group_cols, alpha=float(best_alpha), n_folds=5, seed=RANDOM_SEED
)

if np.isnan(oof_pred_pid).any():
    raise RuntimeError(
        f"OOF prediction_id-level predictions contain NaN (count={np.isnan(oof_pred_pid).sum()})."
    )

tmp_w = train_df.copy()
tmp_w["prediction_id"] = (
    tmp_w["patient_id"].astype(str) + "-" + tmp_w["laterality"].astype(str)
)
pid_counts = (
    tmp_w.groupby("prediction_id", as_index=False)
    .size()
    .rename(columns={"size": "n_images"})
)
pid_truth = tmp_w.groupby("prediction_id", as_index=False)["cancer"].max()
pid_oof = pd.DataFrame(
    {"prediction_id": pid_truth["prediction_id"].values, "oof_pred": oof_pred_pid}
)
pid_fit = pid_truth.merge(pid_oof, on="prediction_id", how="inner").merge(
    pid_counts, on="prediction_id", how="left"
)
pid_fit = pid_fit.sort_values("prediction_id")

y_true_fit = pid_fit["cancer"].astype(np.float64).values
y_pred_fit = pid_fit["oof_pred"].astype(np.float64).values
w_fit = pid_fit["n_images"].fillna(1).astype(np.float64).values

iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
iso.fit(y_pred_fit, y_true_fit, sample_weight=w_fit)

oof_cal_pid = iso.predict(oof_pred_pid)

print(
    "OOF pF1 (uncalibrated, prediction_id-level):",
    probabilistic_f1(y_true_pid, oof_pred_pid),
)
print(
    "OOF pF1 (isotonic-calibrated, prediction_id-level, weighted-fit):",
    probabilistic_f1(y_true_pid, oof_cal_pid),
)

# Change (metric-aligned, minimal): choose a single global scaling k on top of isotonic outputs
# to maximize the same pF1 proxy on OOF prediction_id-level predictions. This preserves core logic
# (same base rates, same isotonic), and only adjusts probability magnitude to better match pF1.
k_grid = np.array([0.5, 0.67, 0.8, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0], dtype=np.float64)
best_k = 1.0
best_k_score = -1.0
for k in k_grid:
    scaled = np.clip(oof_cal_pid * k, 0.0, 1.0)
    sc = probabilistic_f1(y_true_pid, scaled)
    if sc > best_k_score:
        best_k_score = sc
        best_k = float(k)

print("Selected k:", best_k, "with OOF pF1 (isotonic + scaled):", best_k_score)

train_rates = train_df.groupby(group_cols, as_index=False).agg(
    mean_cancer=("cancer", "mean"), cnt=("cancer", "size")
)

global_mean = float(train_df["cancer"].mean())
alpha = float(best_alpha)

train_rates["smoothed"] = (
    train_rates["mean_cancer"] * train_rates["cnt"] + global_mean * alpha
) / (train_rates["cnt"] + alpha)

test_with_rate = test_df.merge(
    train_rates[group_cols + ["smoothed"]], on=group_cols, how="left"
)
test_with_rate["smoothed"] = (
    test_with_rate["smoothed"].fillna(global_mean).astype(np.float64).clip(0.0, 1.0)
)

test_with_rate["smoothed_cal"] = iso.predict(test_with_rate["smoothed"].values)

# Apply the selected global scaling after isotonic calibration (same as OOF selection).
test_with_rate["smoothed_cal"] = (test_with_rate["smoothed_cal"].values * best_k).clip(
    0.0, 1.0
)

pred_by_pid = test_with_rate.groupby("prediction_id", as_index=False)[
    "smoothed_cal"
].mean()

submission = sub_df[["prediction_id"]].merge(
    pred_by_pid, on="prediction_id", how="left"
)

default_cal = float(
    (iso.predict(np.array([global_mean], dtype=np.float64))[0] * best_k).clip(0.0, 1.0)
)
submission["cancer"] = submission["smoothed_cal"].fillna(default_cal).clip(0.0, 1.0)
submission = submission[["prediction_id", "cancer"]]

assert (
    submission.shape[0] == sub_df.shape[0]
), "Row count mismatch vs sample_submission."
assert (
    submission["prediction_id"].values == sub_df["prediction_id"].values
).all(), "prediction_id order mismatch."
assert submission["cancer"].between(0, 1).all(), "Predictions must be within [0,1]."

submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)
print("alpha used:", alpha)
print("k used:", best_k)
print("default_cal used:", default_cal)

OOF pF1 (uncalibrated, prediction_id-level): 0.02011611232409737
OOF pF1 (isotonic-calibrated, prediction_id-level, weighted-fit): 0.020459429633708892
Selected k: 2.0 with OOF pF1 (isotonic + scaled): 0.02722285742103369
  prediction_id    cancer
0       10116_L  0.040385
1       10116_R  0.040385
2       10130_L  0.043821
3       10130_R  0.040385
4       10188_L  0.042676
Wrote submission.csv with shape: (2384, 2)
alpha used: 80.0
k used: 2.0
default_cal used: 0.04038461538461538
